In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("RestaurantBillsPractice")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/28 11:30:19 WARN Utils: Your hostname, ks-VirtualBox, resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/09/28 11:30:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/28 11:30:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
data = spark.read.csv('restaurant_bills_300.csv', header=True, inferSchema=True)
data.printSchema()

root
 |-- total_bill: double (nullable = true)
 |-- tip: double (nullable = true)
 |-- sex: string (nullable = true)
 |-- smoker: string (nullable = true)
 |-- day: string (nullable = true)
 |-- time: string (nullable = true)
 |-- size: integer (nullable = true)



In [3]:
print(f"Total record count: {data.count()}")

Total record count: 300


In [4]:
data.show(5)

+----------+----+------+------+----+------+----+
|total_bill| tip|   sex|smoker| day|  time|size|
+----------+----+------+------+----+------+----+
|       8.0|0.16|Female|   Yes|Thur|Dinner|   1|
|      8.37|1.26|  Male|    No| Sun|Dinner|   4|
|      8.74|0.52|  Male|    No| Sat|Dinner|   1|
|      9.11|1.73|Female|    No| Sat|Dinner|   4|
|      9.48|0.95|  Male|   Yes| Fri|Dinner|   1|
+----------+----+------+------+----+------+----+
only showing top 5 rows


In [5]:
avg_tips = data.groupBy("day").agg(F.avg("tip").alias("avg_tip")).orderBy("day")
avg_tips.show()

+----+-----------------+
| day|          avg_tip|
+----+-----------------+
| Fri|            3.225|
| Sat|           3.5295|
| Sun|3.413599999999999|
|Thur|3.323199999999998|
+----+-----------------+



In [6]:
filter_df = data.filter(F.col("tip") >= F.col("total_bill") *0.05)
print("Filterd record", filter_df.count())          

Filterd record 252


In [7]:
filter_df.show(5) 

+----------+----+------+------+----+------+----+
|total_bill| tip|   sex|smoker| day|  time|size|
+----------+----+------+------+----+------+----+
|      8.37|1.26|  Male|    No| Sun|Dinner|   4|
|      8.74|0.52|  Male|    No| Sat|Dinner|   1|
|      9.11|1.73|Female|    No| Sat|Dinner|   4|
|      9.48|0.95|  Male|   Yes| Fri|Dinner|   1|
|      9.85|2.27|  Male|    No|Thur| Lunch|   5|
+----------+----+------+------+----+------+----+
only showing top 5 rows


In [8]:
data.createOrReplaceTempView("tips_table")

avg_tip_sql = spark.sql(""" SELECT day, AVG(tip) AS avg_tip FROM tips_table GROUP BY day ORDER BY day""")

avg_tip_sql.show()

+----+-----------------+
| day|          avg_tip|
+----+-----------------+
| Fri|            3.225|
| Sat|           3.5295|
| Sun|3.413599999999999|
|Thur|3.323199999999998|
+----+-----------------+



In [9]:
df_results = {row["day"] : row["avg_tip"]
            for row in avg_tips.collect()}

In [10]:
sql_results = {row["day"] : row["avg_tip"]
            for row in avg_tip_sql.collect()}

In [11]:
matches = (df_results.keys() == sql_results.keys()
          and all(
              abs(df_results[day] - sql_results[day]) < 1e-9
              for day in df_results
          )
          )

print("DataFrame API and SQL results match:", matches)

DataFrame API and SQL results match: True
